# 03. RAG — 검색 증강 생성

**RAG(Retrieval-Augmented Generation)** 는 LLM이 답을 생성하기 전에 **외부 문서를 검색**해
관련 정보를 프롬프트에 넣어주는 방식입니다. 모델을 다시 학습하지 않고도 최신·전문 지식을 활용할 수 있습니다.

1. 문서를 임베딩으로 변환
2. FAISS로 벡터 색인·검색
3. 검색 결과를 LLM 프롬프트에 넣어 답변 생성

In [ ]:
import torch
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModel, pipeline
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. 문서 임베딩

검색 대상 문서(지식 베이스)를 임베딩 모델로 벡터화합니다. 여기서는 간단한 예시 문서를 사용합니다.

In [ ]:
documents = [
    'The Eiffel Tower is located in Paris and was completed in 1889.',
    'Python is a programming language created by Guido van Rossum in 1991.',
    'The Great Wall of China is over 13,000 miles long.',
    'Photosynthesis is the process by which plants convert sunlight into energy.',
    'Mount Everest is the highest mountain on Earth at 8,849 meters.',
]

embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embed_tok = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to(device).eval()

def embed(texts):
    enc = embed_tok(texts, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        out = embed_model(**enc)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return F.normalize(emb, p=2, dim=1).cpu().numpy()

doc_embeddings = embed(documents)
print('문서 임베딩 shape:', doc_embeddings.shape)

## 2. FAISS 색인·검색

FAISS는 대량의 벡터에서 유사한 것을 빠르게 찾는 라이브러리입니다.
정규화된 임베딩에 내적(`IndexFlatIP`)을 쓰면 코사인 유사도 검색이 됩니다.

In [ ]:
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

def retrieve(query, k=2):
    q_emb = embed([query])
    scores, idx = index.search(q_emb, k)
    return [(documents[i], float(s)) for i, s in zip(idx[0], scores[0])]

query = 'Who created the Python language?'
hits = retrieve(query)
print('질의:', query)
for doc, score in hits:
    print(f'  ({score:.3f}) {doc}')

## 3. 검색 결과로 답변 생성

검색된 문서를 컨텍스트로 LLM 프롬프트에 넣어 답변을 생성합니다. 이것이 RAG의 핵심 — 검색 + 생성의 결합입니다.

In [ ]:
generator = pipeline('text-generation', model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
                     device=0 if torch.cuda.is_available() else -1,
                     dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

context = '\n'.join(doc for doc, _ in retrieve(query, k=2))
prompt = f'''Use the context to answer the question.

Context:
{context}

Question: {query}
Answer:'''

result = generator(prompt, max_new_tokens=60, do_sample=False)
print(result[0]['generated_text'][len(prompt):].strip())

### 정리

- 문서를 임베딩으로 바꾸고 FAISS로 색인·검색한 뒤, 검색 결과를 LLM 프롬프트에 넣어 답변을 생성했습니다.
- RAG는 모델 재학습 없이 외부·최신 지식을 활용하는 강력한 방법입니다.
- 실제 시스템에서는 문서 분할(chunking), 더 큰 지식 베이스, 정교한 프롬프트 설계로 확장합니다.